# Improvement iteration 4: more facts and self-consistency for the LoRA generator (GPU)

**Question.** Two weaknesses of the Notebook 16 pipeline (dev 610/883): (H1) 72 of its 273 errors have
gold facts outside the top 5, while all-gold recall rises from 86.98% at 5 facts to 93.54% at 10; (H2)
most remaining errors are wrong operations or operands, and 4-beam agreement is too coarse a confidence.

**What changes.** H1: the generator is trained and decoded with 10 facts per input (gold plus random
distractors for training, retrieved top 10 for dev) and 1,024 source tokens. Everything else matches
Notebook 16 (`configs/generator_qwen_lora_top10.toml`). H2: at decode time, 16 programs are sampled per
record and the executed value with most votes wins (self-consistency); its vote share is a label-free
confidence for abstention. Beam sequence scores are saved too.

**Decision this iteration tests.** Whether to replace the Notebook 16 pipeline (610 dev records correct,
0.6908), and whether vote share reaches the M1 abstention target (>= 80% accuracy at >= 50% coverage).

**Protocol.** Train on train; select the epoch by dev greedy execution accuracy; decode beams and
samples on dev with the best adapter. Test is not loaded. Needs a CUDA GPU (Kaggle T4);
`FINEVIDENCE_SMOKE_TEST=1` runs a tiny CPU check.


In [1]:
from __future__ import annotations

import gc
import json
import os
import platform
import random
import subprocess
import time
import tomllib
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import torch
import transformers
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import download_finqa_splits
from finevidence.evaluation.abstention import coverage_curve, majority_vote
from finevidence.generation import (
    build_retrieved_examples,
    build_training_examples,
    load_rankings,
    score_program_prediction,
    select_grounded_candidate,
    summarize_scores,
    verify_rankings_file,
)
from finevidence.generation.causal import (
    IGNORE_INDEX,
    INSTRUCTION,
    PROGRAM_MARKER,
    build_causal_example,
    extract_program,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
# Decode-only rerun: skip training and load the adapter already in the model directory.
REUSE_ADAPTER = os.environ.get("FINEVIDENCE_REUSE_ADAPTER") == "1"
cfg = tomllib.loads((PROJECT_ROOT / "configs" / "generator_qwen_lora_top10.toml").read_text())
assert "test" not in (cfg["train_split"], cfg["evaluation_split"])
pretrained = (
    "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
    if SMOKE_TEST
    else cfg["model"]["pretrained_name"]
)
epochs = 1 if SMOKE_TEST else cfg["training"]["epochs"]
num_beams = 2 if SMOKE_TEST else cfg["generation"]["num_beams"]
num_samples = 4 if SMOKE_TEST else cfg["sampling"]["num_samples"]
suffix = "-smoke" if SMOKE_TEST else ""
model_dir = Path(
    os.environ.get("FINEVIDENCE_MODEL_DIR") or PROJECT_ROOT / cfg["output"]["model_dir"]
)
model_dir = Path(str(model_dir) + suffix)
predictions_dir = PROJECT_ROOT / cfg["output"]["predictions_dir"]
rankings_path = PROJECT_ROOT / cfg["inputs"]["rankings_path"]

seed = cfg["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError(
        "Full training needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check."
    )
use_fp16 = device.type == "cuda"
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "reuse_adapter": REUSE_ADAPTER,
    "pretrained": pretrained,
    "epochs": epochs,
    "num_beams": num_beams,
    "num_samples": num_samples,
    "fp16": use_fp16,
    "seed": seed,
}
print(json.dumps(environment, indent=2))

{
  "commit": "6cc650e002f1851f1801346ad9d17289fb080282",
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "peft": "0.21.2",
  "device": "Tesla T4",
  "smoke_test": false,
  "reuse_adapter": true,
  "pretrained": "Qwen/Qwen2.5-1.5B-Instruct",
  "epochs": 3,
  "num_beams": 4,
  "num_samples": 16,
  "fp16": true,
  "seed": 42
}


## 1. Verified data, rankings and inputs

In [2]:
raw_dir = PROJECT_ROOT / cfg["dataset_root"]
print(download_finqa_splits(raw_dir, (cfg["train_split"], cfg["evaluation_split"])))
verify_rankings_file(rankings_path, cfg["inputs"]["rankings_sha256"])
rankings = load_rankings(rankings_path)
train = load_finqa_split(raw_dir / f"{cfg['train_split']}.json")
dev = load_finqa_split(raw_dir / f"{cfg['evaluation_split']}.json")
if SMOKE_TEST:
    train, dev = train[:24], dev[:8]
facts = cfg["training"]["facts_per_input"]
retrieved_examples = build_retrieved_examples(dev, rankings, facts_per_input=facts)
print({"train_records": len(train), "dev_records": len(dev), "test_loaded": False})

{'train': 'downloaded; verified', 'dev': 'downloaded; verified'}


{'train_records': 6251, 'dev_records': 883, 'test_loaded': False}


## 2. Tokenization

The prompt is the instruction, the source and the marker `program:`; only the program and the
end-of-sequence token are trained (`finevidence.generation.causal`).

In [3]:
tokenizer = AutoTokenizer.from_pretrained(pretrained)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
max_source_tokens = cfg["model"]["max_source_tokens"]
marker_ids = tokenizer(PROGRAM_MARKER, add_special_tokens=False).input_ids


def prompt_ids(example) -> list[int]:
    source = tokenizer(INSTRUCTION + example.source, add_special_tokens=False).input_ids
    return [*source[:max_source_tokens], *marker_ids]


def training_features(example) -> tuple[list[int], list[int]]:
    source = tokenizer(INSTRUCTION + example.source, add_special_tokens=False).input_ids
    target = tokenizer(" " + example.target, add_special_tokens=False).input_ids
    return build_causal_example(
        source, marker_ids, target, tokenizer.eos_token_id, max_source_tokens=max_source_tokens
    )


first = build_training_examples(train, facts_per_input=facts, seed=seed)
lengths = np.array([len(training_features(example)[0]) for example in first])
truncated = sum(
    len(tokenizer(INSTRUCTION + e.source, add_special_tokens=False).input_ids) > max_source_tokens
    for e in first
)
print(
    f"train sequence tokens: median {int(np.median(lengths))}, p95 {int(np.percentile(lengths, 95))}, "
    f"max {lengths.max()}; sources truncated: {truncated}/{len(first)}"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

train sequence tokens: median 479, p95 764, max 1063; sources truncated: 17/6251


## 3. Train LoRA adapters

The base weights stay frozen in fp16; the adapters are kept in fp32 and trained under fp16 autocast with
loss scaling. Each optimizer step averages `micro_batch_size x gradient_accumulation` examples over their
target tokens. After every epoch, dev is decoded greedily and the best adapter is saved.

In [4]:
def clock(seconds: float) -> str:
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m{seconds:02d}s" if hours else f"{minutes}m{seconds:02d}s"


base = AutoModelForCausalLM.from_pretrained(
    pretrained, torch_dtype=torch.float16 if use_fp16 else torch.float32
).to(device)
base.gradient_checkpointing_enable()
base.enable_input_require_grads()
base.config.use_cache = False
lora = cfg["lora"]
model = get_peft_model(
    base,
    LoraConfig(
        r=lora["rank"],
        lora_alpha=lora["alpha"],
        lora_dropout=lora["dropout"],
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    ),
)
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()
model.print_trainable_parameters()


def pad_right(rows: list[list[int]], value: int) -> torch.Tensor:
    width = max(len(row) for row in rows)
    return torch.tensor([row + [value] * (width - len(row)) for row in rows], device=device)


@torch.no_grad()
def decode(
    examples, beams: int, samples: int = 0, batch_size: int | None = None
) -> tuple[list[list[str]], list[list[float]]]:
    """Beam search (samples=0) or sampling; returns programs and length-normalized log-probs."""
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    outputs, all_scores = [], []
    width_per = samples or beams
    batch_size = batch_size or cfg["generation"]["eval_batch_size"]
    for start in range(0, len(examples), batch_size):
        batch = examples[start : start + batch_size]
        prompts = [prompt_ids(example) for example in batch]
        width = max(len(p) for p in prompts)
        input_ids = torch.tensor(
            [[tokenizer.pad_token_id] * (width - len(p)) + p for p in prompts], device=device
        )
        attention = torch.tensor(
            [[0] * (width - len(p)) + [1] * len(p) for p in prompts], device=device
        )
        options = (
            {
                "do_sample": True,
                "num_beams": 1,
                "temperature": cfg["sampling"]["temperature"],
                "top_p": cfg["sampling"]["top_p"],
            }
            if samples
            else {"do_sample": False, "num_beams": beams}
        )
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention,
                max_new_tokens=cfg["model"]["max_new_tokens"],
                num_return_sequences=width_per,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                output_scores=True,
                return_dict_in_generate=True,
                **options,
            )
        new_tokens = generated.sequences[:, width:]
        if beams > 1 and not samples:
            # Beam search already returns the length-normalized log-probability of each sequence.
            sequence_scores = generated.sequences_scores.float().tolist()
        else:
            transition = model.compute_transition_scores(
                generated.sequences, generated.scores, normalize_logits=True
            )
            valid = new_tokens[:, : transition.shape[1]] != tokenizer.pad_token_id
            lengths = valid.sum(dim=1).clamp(min=1)
            sequence_scores = (
                transition.float().masked_fill(~valid, 0).sum(dim=1) / lengths
            ).tolist()
        texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        programs = [extract_program(text) for text in texts]
        for i in range(0, len(programs), width_per):
            outputs.append(programs[i : i + width_per])
            all_scores.append([round(x, 5) for x in sequence_scores[i : i + width_per]])
    model.config.use_cache = False
    return outputs, all_scores


def score(examples, programs) -> tuple[list, dict]:
    scores = [
        score_program_prediction(record, program, example.fact_texts)
        for record, example, program in zip(dev, examples, programs, strict=True)
    ]
    return scores, summarize_scores(scores)


training = cfg["training"]
step_size = training["micro_batch_size"] * training["gradient_accumulation"]
steps_per_epoch = -(-len(train) // step_size)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=training["learning_rate"]
)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(training["warmup_ratio"] * steps_per_epoch * epochs), steps_per_epoch * epochs
)
scaler = torch.amp.GradScaler(enabled=use_fp16)
if REUSE_ADAPTER:
    history = json.loads((model_dir / "history.json").read_text())["history"]
else:
    history = []
best = -1
model_dir.mkdir(parents=True, exist_ok=True)
run_started = time.time()
report_every = max(1, steps_per_epoch // 10)
for epoch in range(0 if REUSE_ADAPTER else epochs):
    model.train()
    examples = build_training_examples(train, facts_per_input=facts, seed=seed + epoch)
    random.Random(seed * 1_000_003 + epoch).shuffle(examples)
    features = [training_features(example) for example in examples]
    started, losses = time.time(), []
    print(f"\nEpoch {epoch + 1}/{epochs}", flush=True)
    for step, start in enumerate(range(0, len(features), step_size), start=1):
        batch = features[start : start + step_size]
        tokens = sum(sum(label != IGNORE_INDEX for label in labels) for _, labels in batch)
        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for chunk_start in range(0, len(batch), training["micro_batch_size"]):
            chunk = batch[chunk_start : chunk_start + training["micro_batch_size"]]
            input_ids = pad_right([ids for ids, _ in chunk], tokenizer.pad_token_id)
            labels = pad_right([labels for _, labels in chunk], IGNORE_INDEX)
            attention = pad_right([[1] * len(ids) for ids, _ in chunk], 0)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
                logits = model(input_ids=input_ids, attention_mask=attention).logits
            shift_logits = logits[:, :-1].float()
            shift_labels = labels[:, 1:]
            loss = (
                torch.nn.functional.cross_entropy(
                    shift_logits.reshape(-1, shift_logits.size(-1)),
                    shift_labels.reshape(-1),
                    ignore_index=IGNORE_INDEX,
                    reduction="sum",
                )
                / tokens
            )
            scaler.scale(loss).backward()
            step_loss += loss.item()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(step_loss)
        if step % report_every == 0 or step == steps_per_epoch:
            elapsed = time.time() - started
            print(
                f"  step {step}/{steps_per_epoch} | loss {np.mean(losses[-report_every:]):.4f} | "
                f"elapsed {clock(elapsed)} | epoch left ~{clock(elapsed / step * (steps_per_epoch - step))}",
                flush=True,
            )
    train_seconds = time.time() - started
    greedy = [candidates[0] for candidates in decode(retrieved_examples, beams=1)[0]]
    _, summary = score(retrieved_examples, greedy)
    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_seconds": round(train_seconds, 1),
            "epoch_seconds_with_dev": round(time.time() - started, 1),
            **{f"dev_greedy_{k}": v for k, v in summary.items() if k != "records"},
        }
    )
    if summary["execution_correct"] > best:
        best = summary["execution_correct"]
        model.save_pretrained(model_dir / "best")
        tokenizer.save_pretrained(model_dir / "best")
    if device.type == "cuda":
        history[-1]["peak_gpu_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
    (model_dir / "history.json").write_text(
        json.dumps({"environment": environment, "history": history}, indent=2)
    )
    print(
        f"Epoch {epoch + 1} done: {history[-1]} | total {clock(time.time() - run_started)}",
        flush=True,
    )
display(pd.DataFrame(history))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


,epoch,train_loss,train_seconds,epoch_seconds_with_dev,dev_greedy_execution_correct,dev_greedy_execution_accuracy,dev_greedy_program_accuracy,dev_greedy_invalid_rate,dev_greedy_operand_grounding_rate,peak_gpu_gb
0,0,0.235735,3595.1,4215.6,556,0.629672,0.579841,0.002265,0.997335,8.76
1,1,0.091895,3611.8,4241.0,590,0.668177,0.617214,0.005663,0.997843,8.76
2,2,0.060353,3616.3,4247.1,604,0.684032,0.640997,0.000000,0.998936,8.76


## 4. Best adapter: beams and self-consistency samples

`top_beam` takes the first of 4 beams, `grounded` the first beam that executes and uses only numbers in
the facts, and `vote` the first of 16 samples whose executed value has the most votes. Sampling decodes one record at a time (16 sequences) to stay within T4 memory. Every candidate,
its length-normalized log-probability and the vote share are saved for the abstention analysis.


In [5]:
# Free the training model before loading the best adapter for decoding.
del model, base, optimizer
gc.collect()
torch.cuda.empty_cache()
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        pretrained, torch_dtype=torch.float16 if use_fp16 else torch.float32
    ),
    model_dir / "best",
).to(device)
model.config.use_cache = False
started = time.time()
beam_candidates, beam_scores = decode(retrieved_examples, beams=num_beams)
print(
    f"decoded {len(beam_candidates)} records with {num_beams} beams in {clock(time.time() - started)}"
)
torch.manual_seed(seed)
started = time.time()
sample_candidates, sample_scores = decode(
    retrieved_examples,
    beams=1,
    samples=num_samples,
    batch_size=cfg["sampling"]["sample_batch_size"],
)
print(
    f"sampled {num_samples} programs for {len(sample_candidates)} records in {clock(time.time() - started)}"
)

votes = [
    majority_vote(options, record["table"])
    for record, options in zip(dev, sample_candidates, strict=True)
]
chosen_by = {
    "top_beam": [options[0] for options in beam_candidates],
    "grounded": [
        options[select_grounded_candidate(options, example.fact_texts, record["table"])]
        for record, example, options in zip(dev, retrieved_examples, beam_candidates, strict=True)
    ],
    "vote": [options[index] for options, (index, _) in zip(sample_candidates, votes, strict=True)],
}
results, scores_by = {}, {}
for selection, chosen in chosen_by.items():
    scores_by[selection], results[f"retrieved_{selection}"] = score(retrieved_examples, chosen)
    path = (
        predictions_dir
        / f"{cfg['experiment_name']}{suffix}_dev_retrieved_{selection}_predictions.jsonl"
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for example, record_score in zip(retrieved_examples, scores_by[selection], strict=True):
            handle.write(
                json.dumps({**vars(record_score), "fact_keys": example.fact_keys, "commit": commit})
                + "\n"
            )
path = predictions_dir / f"{cfg['experiment_name']}{suffix}_dev_candidates.jsonl"
with path.open("w", encoding="utf-8") as handle:
    for example, beams_, bscores, samples_, sscores, (index, share) in zip(
        retrieved_examples,
        beam_candidates,
        beam_scores,
        sample_candidates,
        sample_scores,
        votes,
        strict=True,
    ):
        row = {
            "record_id": example.record_id,
            "fact_keys": example.fact_keys,
            "beam_candidates": beams_,
            "beam_scores": bscores,
            "sample_candidates": samples_,
            "sample_scores": sscores,
            "vote_index": index,
            "vote_share": share,
            "commit": commit,
        }
        handle.write(json.dumps(row) + "\n")
display(pd.DataFrame(results).T)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

decoded 883 records with 4 beams in 23m23s


sampled 16 programs for 883 records in 1h10m23s


,records,execution_correct,execution_accuracy,program_accuracy,invalid_rate,operand_grounding_rate
retrieved_top_beam,883.0,625.0,0.707814,0.663647,0.0,0.997401
retrieved_grounded,883.0,625.0,0.707814,0.663647,0.0,0.997921
retrieved_vote,883.0,617.0,0.698754,0.652322,0.0,0.997335


## 5. Abstention by vote share (dev)

Answer only when the winning value's vote share is at least the threshold; accuracy is measured on the
answered records and always shown with coverage.


In [6]:
curve = coverage_curve(
    [share for _, share in votes], [s.execution_correct for s in scores_by["vote"]]
)
display(pd.DataFrame([vars(point) for point in curve]))

,threshold,answered,coverage,accuracy
0,1.0000,494,0.559456,0.890688
1,0.9375,533,0.603624,0.879925
2,0.8750,575,0.651189,0.866087
3,0.8125,625,0.707814,0.844800
4,0.7500,671,0.759909,0.828614
5,0.6875,703,0.796149,0.812233
6,0.6250,727,0.823330,0.795048
7,0.5625,756,0.856172,0.776455
8,0.5000,788,0.892412,0.761421
9,0.4375,805,0.911665,0.752795
